In [ ]:

import os
import warnings
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

In [7]:
%run utility_function.ipynb

In [ ]:





# ══════════════════════════════════════════════════════════════════════════════
# STEP 4 — Reproduce the exact same 80/20 split
# ══════════════════════════════════════════════════════════════════════════════

feat_cols = get_feat_cols(FEATURE_SET)
missing   = [c for c in feat_cols if c not in master_df.columns]
if missing:
    print(f"⚠️  Dropping {len(missing)} missing columns: {missing}")
    feat_cols = [c for c in feat_cols if c in master_df.columns]

X = master_df[feat_cols]
y = master_df["target"].astype(int)

_, X_test, _, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

print(f"  Held-out test set: {len(X_test)} rows  "
      f"({len(X_test) / len(X) * 100:.0f}% of {len(X)} total)")

# ══════════════════════════════════════════════════════════════════════════════
# STEP 5 — Load model & predict
# ══════════════════════════════════════════════════════════════════════════════

print(f"\nLoading model: {MODEL_PATH}")
# model = joblib.load(MODEL_PATH)
pipeline, feat_cols, team_features, player_features = load_model_bundle(
        model_path,
        fallback_feat_cols = feature_experiments[row['Feature Set']],
        fallback_team      = TEAM_FEATURES_V1,
        fallback_player    = PLAYER_FEATURES_V1,
    )

# preds = model.predict(X_test)
preds = pipeline.predict(X_test)
try:
    proba     = pipeline.predict_proba(X_test)
    prob_away = proba[:, 0]
    prob_draw = proba[:, 1]
    prob_home = proba[:, 2]
except Exception:
    prob_away = prob_draw = prob_home = np.full(len(X_test), np.nan)

# ══════════════════════════════════════════════════════════════════════════════
# STEP 6 — Assemble results CSV
# ══════════════════════════════════════════════════════════════════════════════

results = X_test.copy()
results.insert(0, "ID",            master_df.loc[X_test.index, "ID"].values)
results.insert(1, "actual",        pd.Series(y_test.values).map(LABEL_MAP).values)
results.insert(2, "predicted",     pd.Series(preds).map(LABEL_MAP).values)
results.insert(3, "correct",       (preds == y_test.values).astype(int))
results.insert(4, "prob_home_win", np.round(prob_home, 4))
results.insert(5, "prob_draw",     np.round(prob_draw, 4))
results.insert(6, "prob_away_win", np.round(prob_away, 4))

results.to_csv(OUTPUT_CSV, index=False)

# ══════════════════════════════════════════════════════════════════════════════
# STEP 7 — Summary
# ══════════════════════════════════════════════════════════════════════════════

acc = (preds == y_test.values).mean()
n_wrong = (results["correct"] == 0).sum()

print(f"\n{'='*55}")
print(f"  Accuracy on held-out 20%: {acc:.4f}")
print(f"  Correct  : {results['correct'].sum()} / {len(results)}")
print(f"  Wrong    : {n_wrong} / {len(results)}")
print(f"{'='*55}")
print("\nPer-class accuracy (actual class):")
for cls in [0, 1, 2]:
    mask      = y_test.values == cls
    n_total   = mask.sum()
    n_correct = ((preds == cls) & mask).sum()
    print(f"  {LABEL_MAP[cls]:9s}  {n_correct}/{n_total}  ({n_correct/n_total*100:.1f}% correct)")

print(f"\n✓ Saved → {OUTPUT_CSV}  ({len(results)} rows, {len(results.columns)} columns)")
print("\nExcel tips:")
print("  • Filter  'correct' = 0              → all mispredictions")
print("  • Sort    'prob_<actual_class>' asc   → most overconfident wrong predictions")
print("  • Pivot   'actual' vs 'predicted'     → full confusion matrix")
print("  • Filter  actual='Draw', correct=0   → why draws are missed (usually hardest class)")

Loading training data...
Building feature matrix...
  master_df: 12303 rows × 78 columns
  Held-out test set: 2461 rows  (20% of 12303 total)

Loading model: ../saved_models/NeuralNetwork_Raw__Ratio_Type_13_v2.pkl


NameError: name 'feature_experiments' is not defined

In [13]:
"""
generate_error_analysis.py
──────────────────────────
Reproduces the same 80/20 stratified split used during training
(random_state=42, stratify=y), then runs XGboost_best.pkl only on
the held-out 20% — the data the model never saw during training.

Why 20% only? Because the model was trained on the other 80%.
Predicting on training data would give artificially good results
and hide real failure patterns. This script gives you honest errors.

Usage:
    python generate_error_analysis.py

Output CSV columns:
    ID, actual, predicted, correct (1/0),
    prob_home_win, prob_draw, prob_away_win,
    + all feature columns for slicing in Excel
"""


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG — adjust MODEL_PATH and FEATURE_SET if needed
# ══════════════════════════════════════════════════════════════════════════════

MODEL_PATH  = "../saved_models/NeuralNetwork_Raw__Ratio_Type_13_v2.pkl"
OUTPUT_CSV  = "nn_v2_error_analysis.csv"

DATA_PATHS = {
    "home_team":   "../data/Train_Data/train_home_team_statistics_df.csv",
    "away_team":   "../data/Train_Data/train_away_team_statistics_df.csv",
    "home_player": "../data/Train_Data/train_home_player_statistics_df.csv",
    "away_player": "../data/Train_Data/train_away_player_statistics_df.csv",
    "scores":      "../data/Y_train.csv",
}

# Which feature set was XGboost_best.pkl trained on?
# Check your MLflow run or the notebook output (the "test=0.xxx" lines).
# Options: "raw", "raw_diff", "raw_ratio", "all", "engineered"
FEATURE_SET  = "raw_ratio"    # ← change to match your best model's feature set

# Must match notebook exactly — DO NOT change these
TEST_SIZE    = 0.2
RANDOM_STATE = 42

# ══════════════════════════════════════════════════════════════════════════════
# FEATURES — identical to notebook config
# ══════════════════════════════════════════════════════════════════════════════

TEAM_FEATURES = [
    'TEAM_SHOTS_ON_TARGET_5_last_match_sum',
    'TEAM_GOALS_5_last_match_sum',
    'TEAM_DANGEROUS_ATTACKS_5_last_match_sum',
    'TEAM_GAME_WON_5_last_match_sum',
    'TEAM_GAME_DRAW_5_last_match_sum',
    'TEAM_GAME_LOST_5_last_match_sum',
    'TEAM_SAVES_5_last_match_sum',
    # New features addded on 1March
    'TEAM_INJURIES_5_last_match_sum',
    'TEAM_REDCARDS_5_last_match_sum',
]

PLAYER_FEATURES = [
    "PLAYER_GOALS_5_last_match_average",
    "PLAYER_ASSISTS_5_last_match_average",
    "PLAYER_MINUTES_PLAYED_5_last_match_average",
    "PLAYER_STARTING_LINEUP_5_last_match_average",
    "PLAYER_GOALS_CONCEDED_5_last_match_average",
    "PLAYER_GOALS_season_sum",
    "PLAYER_YELLOWCARDS_5_last_match_sum",
    "PLAYER_GOALS_season_std",
    "PLAYER_PENALTIES_SCORED_season_sum",
    "PLAYER_REDCARDS_season_average",
]

LABEL_MAP = {0: "Away Win", 1: "Draw", 2: "Home Win"}

# FEATURE_SETS = {
#     "raw":        lambda r, d, ra: r,
#     "raw_diff":   lambda r, d, ra: r + d,
#     "raw_ratio":  lambda r, d, ra: r + ra,
#     "all":        lambda r, d, ra: r + d + ra,
#     "engineered": lambda r, d, ra: d + ra,
# }

# # ══════════════════════════════════════════════════════════════════════════════
# # HELPERS
# # ══════════════════════════════════════════════════════════════════════════════

# def prefix_merge(base, df, prefix):
#     rename = {c: f"{prefix}{c}" for c in df.columns if c != "ID"}
#     return base.merge(df.rename(columns=rename), on="ID", how="left")


# def get_feat_cols(feature_set):
#     raw    = ([f"HOME_{c}" for c in TEAM_FEATURES + PLAYER_FEATURES] +
#               [f"AWAY_{c}" for c in TEAM_FEATURES + PLAYER_FEATURES])
#     diffs  = [f"DIFF_{c}"  for c in TEAM_FEATURES + PLAYER_FEATURES]
#     ratios = [f"RATIO_{c}" for c in TEAM_FEATURES + PLAYER_FEATURES]
#     if feature_set not in FEATURE_SETS:
#         raise ValueError(f"Unknown FEATURE_SET '{feature_set}'. Choose from: {list(FEATURE_SETS)}")
#     return FEATURE_SETS[feature_set](raw, diffs, ratios)


# ══════════════════════════════════════════════════════════════════════════════
# STEP 1 — Load data
# ══════════════════════════════════════════════════════════════════════════════

print("Loading training data...")
home_team   = pd.read_csv(DATA_PATHS["home_team"])
away_team   = pd.read_csv(DATA_PATHS["away_team"])
home_player = pd.read_csv(DATA_PATHS["home_player"])
away_player = pd.read_csv(DATA_PATHS["away_player"])
scores      = pd.read_csv(DATA_PATHS["scores"])

# ══════════════════════════════════════════════════════════════════════════════
# STEP 2 — Encode target (same logic as notebook)
# ══════════════════════════════════════════════════════════════════════════════

scores["target"] = np.select(
    [scores["HOME_WINS"] == 1, scores["DRAW"] == 1, scores["AWAY_WINS"] == 1],
    [2, 1, 0],
    default=np.nan,
)

# ══════════════════════════════════════════════════════════════════════════════
# STEP 3 — Build master_df (mirrors notebook exactly)
# ══════════════════════════════════════════════════════════════════════════════

print("Building feature matrix...")

home_player_agg = home_player.groupby("ID")[PLAYER_FEATURES].mean().reset_index()
away_player_agg = away_player.groupby("ID")[PLAYER_FEATURES].mean().reset_index()

master_df = scores[["ID", "target"]].copy()
master_df = prefix_merge(master_df, home_team[["ID"] + TEAM_FEATURES], "HOME_")
master_df = prefix_merge(master_df, home_player_agg,                   "HOME_")
master_df = prefix_merge(master_df, away_team[["ID"] + TEAM_FEATURES], "AWAY_")
master_df = prefix_merge(master_df, away_player_agg,                   "AWAY_")

for col in TEAM_FEATURES + PLAYER_FEATURES:
    h, a = f"HOME_{col}", f"AWAY_{col}"
    if h in master_df.columns and a in master_df.columns:
        master_df[f"DIFF_{col}"]  = master_df[h] - master_df[a]
        master_df[f"RATIO_{col}"] = (master_df[h] + 1) / (master_df[a] + 1)

print(f"  master_df: {master_df.shape[0]} rows × {master_df.shape[1]} columns")
# ══════════════════════════════════════════════════════════════════════════════
# NEW STEP 4 & 5 — Load bundle, align columns, and split
# ══════════════════════════════════════════════════════════════════════════════

print(f"\nLoading model bundle: {MODEL_PATH}")

# 1. Load the model bundle
pipeline, feat_cols, team_features, player_features = load_model_bundle(
    MODEL_PATH,
    fallback_feat_cols = None, # Not needed for v2 bundles
    fallback_team      = TEAM_FEATURES,
    fallback_player    = PLAYER_FEATURES,
)

# 2. Filter master_df to only the features the model expects
missing = [c for c in feat_cols if c not in master_df.columns]
if missing:
    print(f"⚠️  Dropping {len(missing)} missing columns: {missing}")
    feat_cols = [c for c in feat_cols if c in master_df.columns]

X = master_df[feat_cols]
y = master_df["target"].astype(int)

# 3. Reproduce the exact 80/20 split
_, X_test, _, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

print(f"  Held-out test set: {len(X_test)} rows  "
      f"({len(X_test) / len(X) * 100:.0f}% of {len(X)} total)")

# 4. Predict using the pipeline extracted from the bundle
preds = pipeline.predict(X_test)

try:
    proba     = pipeline.predict_proba(X_test)
    prob_away = proba[:, 0]
    prob_draw = proba[:, 1]
    prob_home = proba[:, 2]
except Exception:
    prob_away = prob_draw = prob_home = np.full(len(X_test), np.nan)

Loading training data...
Building feature matrix...
  master_df: 12303 rows × 78 columns

Loading model bundle: ../saved_models/NeuralNetwork_Raw__Ratio_Type_13_v2.pkl
  [bundle] NeuralNetwork  v2  | 57 features | metrics={'train': np.float64(0.48701992234417596), 'val': np.float64(0.4847589362765136), 'test': 0.48476229175132063}
  Held-out test set: 2461 rows  (20% of 12303 total)
